# Phase 4 — Evaluación: Wildfire Segmentation (U-Net ResNet34)

**Entorno**: local, CPU (geoai-wildfire)  
**Input**: `data/patches/` + `models/best_unet_wildfire.pth`  
**Output**: métricas completas + figuras en `results/`

| Métrica | Descripción |
|---------|-------------|
| IoU (fire) | Intersection over Union — clase fuego |
| Precision | De todo lo predicho como fuego, qué % es real |
| Recall | De todo el fuego real, qué % detectamos |
| F1 | Media armónica Precision/Recall |
| AP | Area bajo la curva Precision-Recall |

In [ ]:
import os, random, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
IMG_DIR    = Path('d:/GeoAI/wildfire-spread/data/patches/images')
MASK_DIR   = Path('d:/GeoAI/wildfire-spread/data/patches/masks')
MODEL_PATH = Path('G:/Mon Drive/GeoAI/wildfire-spread/models/best_unet_wildfire.pth')
RESULTS    = Path('G:/Mon Drive/GeoAI/wildfire-spread/results')
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE      = torch.device('cpu')
IN_CHANNELS = 11
VAL_FRAC    = 0.2
THRESHOLD   = 0.5
BEST_IOU    = 0.5014  # registrado en Phase 3

assert MODEL_PATH.exists(), f'Modelo no encontrado: {MODEL_PATH}'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_PATH.name}  ✓')
print(f'Results: {RESULTS}')

In [ ]:
# ── Dataset — idéntico al training ────────────────────────────────────────────
class WildfireDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)
        img[:7]   /= 10000.0
        img[7:10] /= 10000.0
        return (
            torch.from_numpy(img),
            torch.from_numpy(mask).unsqueeze(0)
        )


# ── Recrear el mismo split que en training (SEED=42) ─────────────────────────
img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths), 'Mismatch imágenes/máscaras'
print(f'Patches totales: {len(img_paths)}')

print('Escaneando masks...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0
    for p in tqdm(mask_paths, desc='Fire scan')
], dtype=np.int32)

indices = np.arange(len(img_paths))
_, val_idx = train_test_split(
    indices, test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED
)

val_fire_flags = fire_flags[val_idx]
n_fire_val = val_fire_flags.sum()
print(f'Val: {len(val_idx)} patches  |  {n_fire_val} fuego  |  {len(val_idx)-n_fire_val} sin fuego')

val_ds = WildfireDataset(
    [img_paths[i]  for i in val_idx],
    [mask_paths[i] for i in val_idx]
)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
# ── Cargar modelo ─────────────────────────────────────────────────────────────
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=IN_CHANNELS,
    classes=1,
    activation=None
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo cargado ✓  |  {n_params/1e6:.1f}M parámetros')

In [ ]:
# ── Inferencia completa: pixel-level + per-patch ───────────────────────────────
all_probs   = []
all_targets = []
patch_ious  = []
patch_probs  = []   # guardar para visualizaciones
patch_masks  = []
patch_imgs   = []

print('Corriendo inferencia (CPU, puede tardar 5-10 min)...')
with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='Evaluating'):
        logits = model(imgs.to(DEVICE))               # (B,1,256,256)
        probs  = logits.sigmoid().squeeze(1).numpy()  # (B,256,256)
        masks_np = masks.squeeze(1).numpy()           # (B,256,256)

        all_probs.append(probs.reshape(-1))
        all_targets.append(masks_np.reshape(-1))

        # IoU por patch
        for b in range(len(imgs)):
            p = (probs[b] > THRESHOLD).astype(np.float32)
            m = masks_np[b]
            tp = (p * m).sum()
            fp = (p * (1-m)).sum()
            fn = ((1-p) * m).sum()
            patch_ious.append(float(tp / (tp + fp + fn + 1e-6)))
            patch_probs.append(probs[b])
            patch_masks.append(masks_np[b])
            patch_imgs.append(imgs[b].numpy())

all_probs   = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets).astype(np.int32)
all_preds   = (all_probs > THRESHOLD).astype(np.int32)
patch_ious  = np.array(patch_ious)

print(f'Pixels evaluados     : {len(all_targets):>12,}')
print(f'Pixels fuego (GT)    : {all_targets.sum():>12,}  ({100*all_targets.mean():.2f}%)')
print(f'Pixels pred. fuego   : {all_preds.sum():>12,}  ({100*all_preds.mean():.2f}%)')

In [ ]:
# ── Métricas ──────────────────────────────────────────────────────────────────
precision, recall, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, pos_label=1, average='binary', zero_division=0
)
cm = confusion_matrix(all_targets, all_preds)
tn, fp, fn, tp = cm.ravel()

iou_fire = tp / (tp + fp + fn + 1e-6)
iou_bg   = tn / (tn + fp + fn + 1e-6)
mean_iou = (iou_fire + iou_bg) / 2
accuracy = (tp + tn) / (tp + fp + fn + tn)
ap_score = average_precision_score(all_targets, all_probs)

# IoU sobre patches con fuego solamente
fire_patch_idx = [i for i, f in enumerate(val_fire_flags) if f == 1]
fire_patch_ious = patch_ious[fire_patch_idx]

print('=' * 52)
print('  MÉTRICAS DE EVALUACIÓN — U-Net ResNet34')
print('  Dataset: Corrientes Argentina, Dic 2021–Feb 2022')
print('=' * 52)
print(f'  Precision (fire)   :  {precision:.4f}')
print(f'  Recall (fire)      :  {recall:.4f}')
print(f'  F1-Score (fire)    :  {f1:.4f}')
print(f'  IoU (fire class)   :  {iou_fire:.4f}')
print(f'  IoU (background)   :  {iou_bg:.4f}')
print(f'  Mean IoU           :  {mean_iou:.4f}')
print(f'  Avg Precision (AP) :  {ap_score:.4f}')
print(f'  Accuracy           :  {accuracy:.4f}')
print('─' * 52)
print(f'  Fire-patch IoU mean:  {fire_patch_ious.mean():.4f}')
print(f'  Fire-patch IoU std :  {fire_patch_ious.std():.4f}')
print(f'  Fire-patch IoU >0.5:  {(fire_patch_ious > 0.5).sum()}/{len(fire_patch_ious)}')
print('─' * 52)
print(f'  TP: {tp:>12,}  |  FP: {fp:>12,}')
print(f'  FN: {fn:>12,}  |  TN: {tn:>12,}')
print('=' * 52)

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

labels = ['No fuego', 'Fuego']
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
ax.set_xlabel('Predicción');  ax.set_ylabel('Ground Truth')
ax.set_title('Confusion Matrix (normalizada por fila)')

for i in range(2):
    for j in range(2):
        raw = cm[i, j]
        pct = cm_norm[i, j]
        color = 'white' if pct > 0.5 else 'black'
        ax.text(j, i, f'{pct:.2%}\n({raw:,})', ha='center', va='center',
                color=color, fontsize=9)

plt.tight_layout()
out = RESULTS / 'confusion_matrix.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out}')

In [ ]:
# ── Precision-Recall curve ────────────────────────────────────────────────────
prec_curve, rec_curve, _ = precision_recall_curve(all_targets, all_probs)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rec_curve, prec_curve, color='darkorange', lw=2,
        label=f'U-Net ResNet34 (AP = {ap_score:.3f})')
ax.axhline(all_targets.mean(), color='navy', linestyle='--', lw=1,
           label=f'Baseline (clase positiva = {all_targets.mean():.3f})')
ax.axvline(recall, color='green', linestyle=':', lw=1.5,
           label=f'Threshold 0.5 → Recall={recall:.3f}, Prec={precision:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Clase Fuego')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])

plt.tight_layout()
out = RESULTS / 'pr_curve.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out}')

In [ ]:
# ── Mejores predicciones (top 3 fire patches por IoU) ─────────────────────────
def norm_band(x, p=2):
    vals = x[x > 0]
    if len(vals) == 0:
        return np.zeros_like(x)
    lo, hi = np.percentile(vals, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

best_idx  = sorted(fire_patch_idx, key=lambda i: patch_ious[i], reverse=True)[:3]
worst_idx = sorted(fire_patch_idx, key=lambda i: patch_ious[i])[:3]

def plot_panel(indices, title, filename):
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row, vi in enumerate(indices):
        img_np   = patch_imgs[vi]
        mask_np  = patch_masks[vi]
        prob_np  = patch_probs[vi]
        pred_bin = (prob_np > THRESHOLD).astype(np.uint8)
        iou_v    = patch_ious[vi]

        rgb = np.dstack([norm_band(img_np[2]), norm_band(img_np[1]), norm_band(img_np[0])])

        # Error map: TP=green, FP=orange, FN=red
        error = np.zeros((*mask_np.shape, 3))
        error[(pred_bin==1) & (mask_np==1)] = [0.0, 0.8, 0.0]  # TP verde
        error[(pred_bin==1) & (mask_np==0)] = [1.0, 0.5, 0.0]  # FP naranja
        error[(pred_bin==0) & (mask_np==1)] = [0.9, 0.1, 0.1]  # FN rojo

        overlay = rgb.copy()
        overlay[pred_bin == 1] = [1.0, 0.25, 0.0]

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f'RGB  |  IoU={iou_v:.3f}'); axes[row, 0].axis('off')

        axes[row, 1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1)
        axes[row, 1].set_title('Ground truth'); axes[row, 1].axis('off')

        axes[row, 2].imshow(prob_np, cmap='hot', vmin=0, vmax=1)
        axes[row, 2].set_title('Predicción (prob)'); axes[row, 2].axis('off')

        axes[row, 3].imshow(error)
        axes[row, 3].set_title('TP=verde | FP=naranja | FN=rojo')
        axes[row, 3].axis('off')

    tp_patch = mpatches.Patch(color=[0,0.8,0], label='TP (detectado ✓)')
    fp_patch = mpatches.Patch(color=[1,0.5,0], label='FP (falsa alarma)')
    fn_patch = mpatches.Patch(color=[0.9,0.1,0.1], label='FN (fuego perdido)')
    fig.legend(handles=[tp_patch, fp_patch, fn_patch], loc='lower center',
               ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    out = RESULTS / filename
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Guardado: {out}')


plot_panel(best_idx,  'Mejores predicciones (Top 3 IoU)',   'best_predictions.png')
plot_panel(worst_idx, 'Peores predicciones (Bottom 3 IoU)', 'worst_predictions.png')

In [ ]:
# ── Distribución IoU sobre patches con fuego ──────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(fire_patch_ious, bins=20, color='firebrick', edgecolor='white', alpha=0.85)
ax.axvline(fire_patch_ious.mean(), color='black', lw=1.5, linestyle='--',
           label=f'Media: {fire_patch_ious.mean():.3f}')
ax.axvline(0.5, color='green', lw=1.5, linestyle=':',
           label='Umbral 0.5')
ax.set_xlabel('IoU'); ax.set_ylabel('Patches')
ax.set_title(f'Distribución IoU — {len(fire_patch_ious)} patches con fuego')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = RESULTS / 'iou_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out}')

In [ ]:
# ── Resumen para README ───────────────────────────────────────────────────────
print('\n' + '='*60)
print('  BLOQUE PARA README.md')
print('='*60)
print(f'''
## Results

| Metric | Value |
|--------|-------|
| **IoU (fire class)** | **{iou_fire:.4f}** |
| Precision | {precision:.4f} |
| Recall | {recall:.4f} |
| F1-Score | {f1:.4f} |
| Average Precision (AP) | {ap_score:.4f} |
| Mean IoU (fire + bg) | {mean_iou:.4f} |

> Model: U-Net ResNet34 | Dataset: Corrientes, Argentina (Dec 2021–Feb 2022)
> Training: 40 epochs, A100 GPU | Best Val IoU: {BEST_IOU:.4f} (epoch 16)
''')
print('='*60)

print(f'\nFiguras guardadas en: {RESULTS}')
print('  - confusion_matrix.png')
print('  - pr_curve.png')
print('  - best_predictions.png')
print('  - worst_predictions.png')
print('  - iou_distribution.png')
print('  - training_curves.png   (Phase 3)')
print('  - predictions_sample.png  (Phase 3)')